# Clean Greater Sydney sales and LGA factors

Build the canonical 34-LGA lookup and a quarterly sales table from the original DCJ workbooks. Keep the published Greater Sydney and ring aggregates and use the first-published median for each quarter (D06). The table retains Wollondilly's missing 2019 unit values (D05).

The notebook also builds four D07 factors from the raw boundaries, GPO point, population estimates, Census income and DCJ rent tables. Run from the repository root with the task's nbconvert command.

## 1. Read the DCJ workbooks

Read the `LGA` sheet and keep the columns needed for sales and revisions. Find the heading by its dwelling-type label because its Excel row varies. Read the quarter from the title in older files or the reporting-period line in newer files. Keep the original worksheet row number so exported values can be checked by hand.

In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display
from openpyxl import load_workbook

repo_root = Path("..") if Path.cwd().name == "notebooks" else Path(".")
raw_dir = repo_root / "data/raw"

raw_columns = [
    "gmr", "sydney_group", "ring", "lga", "source_dwelling",
    "first_quartile", "median_thousands", "third_quartile", "mean_thousands",
    "count_raw", "quarterly_change_raw", "annual_change_raw",
]

def read_sales_workbook(path):
    """Read one issue's reporting quarter and LGA table with Excel row numbers."""
    workbook = load_workbook(path, read_only=True, data_only=True)
    try:
        rows = list(workbook["LGA"].iter_rows(max_col=12, values_only=True))
    finally:
        workbook.close()

    caption = rows[0][0]
    if str(rows[1][0]).startswith("Reporting period:"):
        caption = rows[1][0]
    quarter_month = " ".join(caption.replace(" Quarter", "").split()[-2:])
    quarter_end_month = pd.to_datetime(quarter_month, format="%B %Y")
    quarter_date = quarter_end_month.to_period("Q").start_time

    header_index = next(index for index, row in enumerate(rows) if row[4] == "DwellingType")
    table = pd.DataFrame(rows[header_index + 1:], columns=raw_columns, dtype=object)
    first_data_row = header_index + 2
    table["source_row"] = range(first_data_row, first_data_row + len(table))
    table["source_file"] = path.name
    table["quarter_date"] = quarter_date
    table = table.loc[table["source_dwelling"].notna()]
    return table, header_index + 1

workbooks = sorted((raw_dir / "dcj_sales").glob("*.xlsx"))
tables = []
headers = []
for path in workbooks:
    table, header_row = read_sales_workbook(path)
    tables.append(table)
    headers.append(header_row)

raw_sales = pd.concat(tables, ignore_index=True)
print(f"{len(workbooks)} workbooks, {len(raw_sales):,} rows in their LGA tables.")
print("Workbooks by Excel header row")
display(pd.Series(headers, name="header_row").value_counts().sort_index().rename("workbooks"))

29 workbooks, 81,127 rows in their LGA tables.
Workbooks by Excel header row


header_row
5     2
6     1
7    26
Name: workbooks, dtype: int64

## 2. Select published Greater Sydney rows

Use column A = `GMR (Greater Metropolitan Region)` and B = `Greater Sydney`. Within that block, C identifies each ring. D gives the LGA name on an LGA row and `Total` on a ring aggregate row. The published Greater Sydney aggregate has `Total` in both C and D.

Keep column E's `Non Strata` and `Strata` labels as house and unit respectively. Use the source aggregate values directly as benchmarks.

In [2]:
rings = ["Inner Ring", "Middle Ring", "Outer Ring"]
dwelling_labels = {"Non Strata": "house", "Strata": "unit"}
sydney_block = raw_sales.loc[
    raw_sales["gmr"].eq("GMR (Greater Metropolitan Region)")
    & raw_sales["sydney_group"].eq("Greater Sydney")
]
named_ring = sydney_block["ring"].isin(rings)
greater_total = sydney_block["ring"].eq("Total") & sydney_block["lga"].eq("Total")
selected = sydney_block.loc[
    (named_ring | greater_total) & sydney_block["source_dwelling"].isin(dwelling_labels)
].copy()

selected["geo_level"] = "lga"
selected.loc[selected["lga"].eq("Total"), "geo_level"] = "ring"
selected.loc[selected["ring"].eq("Total"), "geo_level"] = "greater_sydney"
selected["geo_name"] = selected["lga"]
selected.loc[selected["geo_level"].eq("ring"), "geo_name"] = selected["ring"]
selected.loc[selected["geo_level"].eq("greater_sydney"), "geo_name"] = "Greater Sydney"
selected["region"] = selected["ring"].where(selected["geo_level"].ne("greater_sydney"))
selected["dwelling_type"] = selected["source_dwelling"].map(dwelling_labels)
print("Source rows retained before completing the grid")
display(selected.groupby("geo_level").size().rename("rows"))

Source rows retained before completing the grid


geo_level
greater_sydney      58
lga               1971
ring               174
Name: rows, dtype: int64

## 3. Convert values and record suppression

Column G is the median in thousands of dollars and J is the number of sales. Strip commas from numeric text, and represent a dash or small-sample marker as missing in the numeric fields. Keep the original marker meaning in a status field. Read the quarterly change from K and the annual change from L for the D08 sensitivity check. Excel numeric changes are already fractions, and text percentages are divided by 100.

In [3]:
def parse_number(values):
    """Read an Excel measure with numeric text, commas or DCJ markers."""
    text = values.astype("string").str.strip().str.replace(",", "", regex=False)
    return pd.to_numeric(text.mask(text.isin(["", "-", "s"])), errors="raise")

selected["median_price"] = (parse_number(selected["median_thousands"]) * 1000).astype("Int64")
selected["sales_count"] = parse_number(selected["count_raw"]).astype("Int64")
selected["price_status"] = "published"
selected.loc[selected["median_thousands"].eq("-"), "price_status"] = "suppressed"
selected["count_status"] = "published"
selected.loc[selected["count_raw"].eq("-"), "count_status"] = "suppressed_10_or_fewer"
selected.loc[selected["count_raw"].eq("s"), "count_status"] = "suppressed_30_or_fewer"

assert selected["median_price"].isna().eq(selected["price_status"].ne("published")).all()
assert selected["sales_count"].isna().eq(selected["count_status"].ne("published")).all()

change_columns = ["quarterly_change", "annual_change"]
for column in change_columns:
    change_text = selected[f"{column}_raw"].astype("string").str.strip()
    percent_text = change_text.str.endswith("%", na=False)
    change = parse_number(change_text.str.removesuffix("%"))
    change.loc[percent_text] = change.loc[percent_text] / 100
    assert change.isna().eq(change_text.fillna("").isin(["", "-"])).all()
    scaled = change.dropna() * 10_000
    assert (scaled - scaled.round()).abs().lt(1e-7).all()
    selected[column] = change.round(4)

print("Source numeric values retained")
display(selected[["median_price", "sales_count", *change_columns]].count())

Source numeric values retained


median_price        2188
sales_count         2058
quarterly_change    2188
annual_change       2188
dtype: int64

## 4. Complete the quarterly grid

Make one expected row for every combination of workbook quarter, geography and house or unit. Attach the published values and label an unmatched row `not_in_source`. Keep its workbook name for traceability and leave its Excel row blank. Flag a row when DCJ withholds the sales count or the row is absent.

In [4]:
keys = ["quarter_date", "geo_level", "geo_name", "dwelling_type"]
quarters = raw_sales[["quarter_date", "source_file"]].drop_duplicates()
geographies = selected[["geo_level", "geo_name", "region"]].drop_duplicates()
dwellings = pd.DataFrame({"dwelling_type": list(dwelling_labels.values())})

grid = quarters.merge(geographies, how="cross").merge(dwellings, how="cross")
source_values = keys + [
    "median_price", "sales_count", "price_status", "count_status", "source_row", *change_columns,
]
sales = grid.merge(selected[source_values], on=keys, how="left", validate="one_to_one", indicator=True)
not_in_source = sales.pop("_merge").eq("left_only")
sales.loc[not_in_source, ["price_status", "count_status"]] = "not_in_source"
sales["source_row"] = sales["source_row"].astype("Int64")
sales["low_sales_flag"] = sales["count_status"].ne("published")
sales = sales.sort_values(keys, ignore_index=True)

assert sales["source_row"].notna().sum() == len(selected)
missing_changes = {}
for column in change_columns:
    source_cells = selected[f"{column}_raw"].astype("string").str.strip()
    reasons = {
        "dash in source": int(source_cells.eq("-").sum()),
        "blank in source": int(source_cells.fillna("").eq("").sum()),
        "row absent": int(not_in_source.sum()),
    }
    assert sales[column].isna().sum() == sum(reasons.values())
    missing_changes[column] = reasons

print("Missing reported sales changes by source reason")
display(pd.DataFrame(missing_changes))
print("Rows added for combinations absent from the workbooks")
display(sales.loc[sales["price_status"].eq("not_in_source"), [
    "quarter_date", "geo_name", "dwelling_type", "source_file",
]])

Missing reported sales changes by source reason


,quarterly_change,annual_change
dash in source,15,15
blank in source,0,0
row absent,1,1


Rows added for combinations absent from the workbooks


,quarter_date,geo_name,dwelling_type,source_file
143,2019-04-01,Wollondilly,unit,issue-129-sales-tables-june-2019.xlsx


## 5. Build the LGA lookup

Use the canonical DCJ label and ring from the raw LGA rows. Check every quarter's ring assignment. Match the six spelling differences explicitly to the ABS labels in ERP Table 1, then retain the five-digit LGA 2025 code for later joins. Check that each code also occurs in the T02 boundary export.

In [5]:
dcj_lgas = selected.loc[selected["geo_level"].eq("lga"), ["quarter_date", "geo_name", "region"]]
membership = dcj_lgas.drop_duplicates().rename(columns={"geo_name": "lga"})
per_lga = membership.groupby("lga").agg(quarters=("quarter_date", "nunique"), rings=("region", "nunique"))
assert len(per_lga) == 34
assert per_lga["quarters"].eq(29).all() and per_lga["rings"].eq(1).all()

dcj_to_abs_name = {
    "Bayside": "Bayside (NSW)",
    "Campbelltown": "Campbelltown (NSW)",
    "Central Coast": "Central Coast (NSW)",
    "Ku-Ring-Gai": "Ku-ring-gai",
    "Sutherland Shire": "Sutherland",
    "The Hills Shire": "The Hills",
}
lga_lookup = membership[["lga", "region"]].drop_duplicates()
lga_lookup["abs_name_2025"] = lga_lookup["lga"].replace(dcj_to_abs_name)
assert dcj_to_abs_name.keys() <= set(lga_lookup["lga"])

erp = pd.read_excel(
    raw_dir / "abs_erp/32180DS0004_2001-25.xlsx", sheet_name="Table 1",
    header=None, skiprows=6, usecols="A:B",
    names=["lga_code_2025", "abs_name_2025"], dtype="string",
)
erp = erp.loc[erp["lga_code_2025"].str.fullmatch(r"\d{5}", na=False)]
lga_lookup = lga_lookup.merge(erp, on="abs_name_2025", how="left", validate="one_to_one")
lookup_columns = ["lga", "region", "lga_code_2025", "abs_name_2025"]
lga_lookup = lga_lookup[lookup_columns].sort_values("lga", ignore_index=True)
codes = set(lga_lookup["lga_code_2025"])
assert len(lga_lookup) == len(codes) == 34
assert lga_lookup["lga_code_2025"].str.fullmatch(r"\d{5}", na=False).all()
assert codes <= set(erp["lga_code_2025"])

with (raw_dir / "abs_asgs/LGA_2025_Sydney_region.geojson").open(encoding="utf-8") as stream:
    features = json.load(stream)["features"]
boundary_codes = {feature["properties"]["lga_code_2025"] for feature in features}
assert codes <= boundary_codes
ring_counts = lga_lookup["region"].value_counts().reindex(rings)
assert ring_counts.to_dict() == {"Inner Ring": 8, "Middle Ring": 12, "Outer Ring": 14}
print("No LGA changed rings across the 29 quarters. All 34 codes occur in ERP and GeoJSON.")
display(ring_counts.rename("lgas"))
display(lga_lookup.loc[lga_lookup["lga"].isin(dcj_to_abs_name)].reset_index(drop=True))

No LGA changed rings across the 29 quarters. All 34 codes occur in ERP and GeoJSON.


region
Inner Ring      8
Middle Ring    12
Outer Ring     14
Name: lgas, dtype: int64

,lga,region,lga_code_2025,abs_name_2025
0,Bayside,Middle Ring,10500,Bayside (NSW)
1,Campbelltown,Outer Ring,11500,Campbelltown (NSW)
2,Central Coast,Outer Ring,11650,Central Coast (NSW)
3,Ku-Ring-Gai,Middle Ring,14500,Ku-ring-gai
4,Sutherland Shire,Outer Ring,17150,Sutherland
5,The Hills Shire,Outer Ring,17420,The Hills


## 6. Measure revisions (D06)

Compare each later issue with the first-published median for the immediately preceding quarter. Sort the geography and dwelling-type series by quarter before shifting one row. The completed grid preserves that alignment when a source row is missing. Count only pairs with both medians and a usable quarterly change. Analyse LGAs and Greater Sydney separately.

Let `A` be the current published median in dollars, `B` the first-published previous median in dollars and `q = 1 + change` the published quarterly ratio. The implied revised prior median is `A / q`. The point estimate of the revision is `A / (q × B) − 1`.

All selected medians are published in whole thousands of dollars, and all published changes can be expressed to 0.01 percentage point. Under nearest rounding, allow each median ±$500 and allow `q` ±0.00005. If the underlying prior median were unchanged, it would lie in `[B − 500, B + 500]`. The smallest and largest compatible displayed current medians would then be `(B − 500) × (q − 0.00005) − 500` and `(B + 500) × (q + 0.00005) + 500`. Divide both endpoints by `q × B` and subtract one to get the rounding-only revision bounds for that pair. This uses the precision of both prices and the change. Smaller medians have a wider percentage bound.

In [6]:
comparison_levels = ["lga", "greater_sydney"]
comparisons = sales.loc[sales["geo_level"].isin(comparison_levels)].copy()
series_keys = ["geo_level", "geo_name", "dwelling_type"]
previous_columns = ["quarter_date", "median_price", "sales_count", "source_file", "source_row"]
series = comparisons.sort_values("quarter_date").groupby(series_keys)
previous = series[previous_columns].shift(1).add_prefix("previous_")
comparisons = comparisons.join(previous)
comparisons = comparisons.loc[comparisons["previous_quarter_date"].notna()].copy()
quarter_numbers = comparisons["quarter_date"].dt.to_period("Q").astype("int64")
previous_numbers = comparisons["previous_quarter_date"].dt.to_period("Q").astype("int64")
assert quarter_numbers.sub(previous_numbers).eq(1).all()

known_changes = selected["quarterly_change"].dropna()
assert (known_changes * 10_000).sub((known_changes * 10_000).round()).abs().lt(1e-7).all()
assert selected["median_price"].dropna().mod(1000).eq(0).all()

price_pairs = comparisons.dropna(subset=["median_price", "previous_median_price"])
revisions = price_pairs.dropna(subset=["quarterly_change"]).copy()
price_rounding = 500
change_rounding = 0.00005

ratio = 1 + revisions["quarterly_change"]
assert ratio.gt(change_rounding).all()
prior_price = revisions["previous_median_price"]
revisions["implied_revised_prev"] = revisions["median_price"] / ratio
revisions["revision_pct"] = (revisions["implied_revised_prev"] / prior_price - 1) * 100
revisions["absolute_revision_pct"] = revisions["revision_pct"].abs()

minimum_display = (prior_price - price_rounding) * (ratio - change_rounding) - price_rounding
maximum_display = (prior_price + price_rounding) * (ratio + change_rounding) + price_rounding
lower_bound_pct = (minimum_display / (ratio * prior_price) - 1) * 100
upper_bound_pct = (maximum_display / (ratio * prior_price) - 1) * 100
revisions["within_rounding"] = revisions["revision_pct"].between(lower_bound_pct, upper_bound_pct)
revisions["max_rounding_pct"] = pd.concat([lower_bound_pct.abs(), upper_bound_pct.abs()], axis=1).max(axis=1)

summary_keys = ["geo_level", "dwelling_type"]
revision_summary = revisions.groupby(summary_keys).agg(
    compared=("revision_pct", "size"),
    median_abs_pct=("absolute_revision_pct", "median"),
    p90_abs_pct=("absolute_revision_pct", lambda values: values.quantile(0.90)),
    max_abs_pct=("absolute_revision_pct", "max"),
    within_rounding_pct=("within_rounding", "mean"),
    max_rounding_pct=("max_rounding_pct", "max"),
)
revision_summary["within_rounding_pct"] *= 100
revision_summary.insert(0, "possible", comparisons.groupby(summary_keys).size())
print("Revision and rounding figures are in percent.")
display(revision_summary.round(4).T)

Revision and rounding figures are in percent.


geo_level           greater_sydney               lga         
dwelling_type                house     unit    house     unit
possible                        28       28      952      952
compared                        28       28      952      932
median_abs_pct               1.848   0.4475   0.5747   0.2868
p90_abs_pct                 2.8272   0.8399    3.976   2.3344
max_abs_pct                 3.5991   1.2821  39.1635   18.741
within_rounding_pct         7.1429  28.5714  26.9958  42.8112
max_rounding_pct            0.1199   0.1487   0.1805   0.2487

### Largest revisions and relevance to the project windows

Show the five largest absolute revisions for LGAs and for Greater Sydney. Keep the sign, the implied median and the worksheet references. Check which revisions affect quarters included in the 2019 baseline or in the latest four-quarter window. This comparison uses revisions within those windows and leaves the growth rankings for the later analysis task.

In [7]:
sorted_revisions = revisions.sort_values("absolute_revision_pct", ascending=False, kind="stable")
for level in comparison_levels:
    largest = sorted_revisions.loc[sorted_revisions["geo_level"].eq(level)].head(5).copy()
    largest.index = range(1, len(largest) + 1)
    print(level, "five largest absolute revisions")
    display(largest[[
        "geo_name", "dwelling_type", "previous_quarter_date", "previous_median_price",
        "implied_revised_prev", "revision_pct",
    ]].round({"implied_revised_prev": 0, "revision_pct": 3}))

    prior_cell = "G" + largest["previous_source_row"].astype("string")
    current_row = largest["source_row"].astype("string")
    display(pd.DataFrame({
        "previous_issue": largest["previous_source_file"],
        "previous_cell": "LGA!" + prior_cell,
        "current_issue": largest["source_file"],
        "current_cells": "LGA!G" + current_row + ", K" + current_row,
    }))

latest_four_quarters = set(quarters["quarter_date"].sort_values().tail(4))
lga_revisions = revisions.loc[revisions["geo_level"].eq("lga")]
for period, window in [
    ("2019 baseline", lga_revisions["previous_quarter_date"].dt.year.eq(2019)),
    ("Latest four quarters", lga_revisions["previous_quarter_date"].isin(latest_four_quarters)),
]:
    in_window = lga_revisions.loc[window]
    print(period, "revisable LGA quarters through", in_window["previous_quarter_date"].max().date())
    display(in_window.groupby("dwelling_type")["absolute_revision_pct"].agg(
        pairs="size", median_abs_pct="median", max_abs_pct="max",
    ).round(3))

lga five largest absolute revisions


,geo_name,dwelling_type,previous_quarter_date,previous_median_price,implied_revised_prev,revision_pct
1,Woollahra,house,2025-04-01,4958000,6899724.0,39.163
2,Woollahra,house,2022-10-01,4175000,5300036.0,26.947
3,Woollahra,house,2023-04-01,4240000,5349929.0,26.178
4,Woollahra,house,2021-10-01,4250000,5299685.0,24.698
5,Mosman,house,2023-04-01,4300000,5312532.0,23.547


,previous_issue,previous_cell,current_issue,current_cells
1,sales_tables_june_2025_quarter.xlsx,LGA!G1796,sales_tables_september_2025_quarter.xlsx,"LGA!G1824, K1824"
2,issue-143-sales-tables-december-2022.xlsx,LGA!G1796,issue-144-sales-tables-march-2023.xlsx,"LGA!G1800, K1800"
3,issue-145-sales-tables-june-2023.xlsx,LGA!G1808,issue-146-sales-tables-september-2023.xlsx,"LGA!G1808, K1808"
4,issue-139-sales-tables-december-2021.xlsx,LGA!G1828,issue-140-sales-tables-march-2022.xlsx,"LGA!G1808, K1808"
5,issue-145-sales-tables-june-2023.xlsx,LGA!G1793,issue-146-sales-tables-september-2023.xlsx,"LGA!G1793, K1793"


greater_sydney five largest absolute revisions


,geo_name,dwelling_type,previous_quarter_date,previous_median_price,implied_revised_prev,revision_pct
1,Greater Sydney,house,2021-01-01,1110000,1149950.0,3.599
2,Greater Sydney,house,2024-10-01,1452000,1500052.0,3.309
3,Greater Sydney,house,2019-01-01,865000,890032.0,2.894
4,Greater Sydney,house,2025-01-01,1430000,1470021.0,2.799
5,Greater Sydney,house,2024-04-01,1460000,1500265.0,2.758


,previous_issue,previous_cell,current_issue,current_cells
1,issue-136-sales-tables-march-2021.xlsx,LGA!G1695,issue-137-sales-tables-june-2021.xlsx,"LGA!G1695, K1695"
2,issue-151-sales-tables-dec-2024.xlsx,LGA!G1679,issue-152-sales-tables-mar-2025.xlsx,"LGA!G1671, K1671"
3,issue-128-sales-tables-march-2019.xlsx,LGA!G1649,issue-129-sales-tables-june-2019.xlsx,"LGA!G1647, K1647"
4,issue-152-sales-tables-mar-2025.xlsx,LGA!G1671,sales_tables_june_2025_quarter.xlsx,"LGA!G1667, K1667"
5,issue-149-sales-tables-june-2024-quarter.xlsx,LGA!G1675,Sales_tables_September_2024_quarter.xlsx,"LGA!G1675, K1675"


2019 baseline revisable LGA quarters through 2019-10-01


,pairs,median_abs_pct,max_abs_pct
dwelling_type,,,
house,136,0.613,13.287
unit,132,0.467,8.925


Latest four quarters revisable LGA quarters through 2025-10-01


,pairs,median_abs_pct,max_abs_pct
dwelling_type,,,
house,102,0.717,39.163
unit,102,0.285,11.001


### Limitations wording

DCJ can revise quarterly medians after release, so this project consistently uses first-published values. The median absolute implied revision was 0.57% for LGA houses and 0.29% for units, with a 39.16% house outlier in the latest four-quarter window. These revisions could move close rankings and affect LGAs with unusually large revised quarters.

## 7. Validate the sales table

Check the complete grid, quarter coverage, geography lookup, missing-value statuses and plausible price range. Compare six values or statuses with the cells reviewed in T01. Then reconcile with the marker counts from `01_explore.ipynb` section 6, which included the all-dwelling `Total` rows and, for its full-sheet count, repeated geography levels.

In [8]:
expected_counts = {"greater_sydney": 58, "lga": 1972, "ring": 174}
assert sales.groupby("geo_level").size().to_dict() == expected_counts
assert not sales.duplicated(keys).any()

expected_quarters = pd.date_range("2019-01-01", "2026-01-01", freq="QS")
actual_quarters = pd.DatetimeIndex(quarters["quarter_date"].sort_values())
assert actual_quarters.equals(expected_quarters)
lga_sales = sales.loc[sales["geo_level"].eq("lga")]
assert set(lga_sales["geo_name"]) == set(lga_lookup["lga"])

absent = sales.loc[sales["price_status"].eq("not_in_source")]
assert len(absent) == 1
assert absent[keys].iloc[0].tolist() == [pd.Timestamp("2019-04-01"), "lga", "Wollondilly", "unit"]
assert sales["source_row"].isna().eq(sales["price_status"].eq("not_in_source")).all()
assert sales["source_row"].isna().eq(sales["count_status"].eq("not_in_source")).all()
assert sales["source_file"].notna().all()
assert sales["median_price"].isna().eq(sales["price_status"].ne("published")).all()
assert sales["sales_count"].isna().eq(sales["count_status"].ne("published")).all()
assert sales["low_sales_flag"].eq(sales["count_status"].ne("published")).all()
assert sales["median_price"].dropna().between(100_000, 10_000_000).all()
assert sales["sales_count"].dropna().gt(0).all()
assert sales["region"].isna().eq(sales["geo_level"].eq("greater_sydney")).all()

checks = [
    ("2026-01-01", "lga", "Blacktown", "house", "median_price", 1_250_000),
    ("2026-01-01", "lga", "Sydney", "unit", "median_price", 967_000),
    ("2026-01-01", "lga", "Parramatta", "unit", "sales_count", 835),
    ("2026-01-01", "greater_sydney", "Greater Sydney", "house", "median_price", 1_550_000),
    ("2026-01-01", "greater_sydney", "Greater Sydney", "unit", "median_price", 855_000),
    ("2019-01-01", "lga", "Wollondilly", "unit", "price_status", "suppressed"),
]
indexed_sales = sales.set_index(keys)
for date, level, name, dwelling, field, expected in checks:
    result = indexed_sales.loc[(pd.Timestamp(date), level, name, dwelling)]
    assert result[field] == expected, (date, name, dwelling, field, result[field])
    column = "J" if field == "sales_count" else "G"
    print(f"{date} {name} {dwelling}: {expected} ({result['source_file']}, LGA!{column}{result['source_row']})")

print("Rows by geography", expected_counts)
print("All key, date, missing-value, price-range and spot checks passed.")
print("LGA price statuses")
display(lga_sales.groupby(["dwelling_type", "price_status"]).size().unstack(fill_value=0))
print("LGA count statuses")
display(lga_sales.groupby(["dwelling_type", "count_status"]).size().unstack(fill_value=0))
print("Published price minimum and maximum in dollars",
      sales["median_price"].min(), sales["median_price"].max())

2026-01-01 Blacktown house: 1250000 (sales-tables-march-2026-quarter.xlsx, LGA!G1865)
2026-01-01 Sydney unit: 967000 (sales-tables-march-2026-quarter.xlsx, LGA!G1815)
2026-01-01 Parramatta unit: 835 (sales-tables-march-2026-quarter.xlsx, LGA!J1851)
2026-01-01 Greater Sydney house: 1550000 (sales-tables-march-2026-quarter.xlsx, LGA!G1691)
2026-01-01 Greater Sydney unit: 855000 (sales-tables-march-2026-quarter.xlsx, LGA!G1692)
2019-01-01 Wollondilly unit: suppressed (issue-128-sales-tables-march-2019.xlsx, LGA!G1863)
Rows by geography {'greater_sydney': 58, 'lga': 1972, 'ring': 174}
All key, date, missing-value, price-range and spot checks passed.
LGA price statuses


price_status,not_in_source,published,suppressed
dwelling_type,,,
house,0,986,0
unit,1,970,15


LGA count statuses


count_status,not_in_source,published,suppressed_10_or_fewer,suppressed_30_or_fewer
dwelling_type,,,,
house,0,949,0,37
unit,1,877,15,93


Published price minimum and maximum in dollars 404000 6750000


### Reconcile with the raw marker counts

Recount the source markers at the same scope as the T01 audit. Compare the Sydney LGA counts before and after removing the `Total` dwelling rows. Published ring and Greater Sydney rows are part of the export, so check those for additional markers. The explicitly added absent row carries no raw marker.

In [9]:
def suppression_counts(table):
    """Count the original median and sales-count markers without changing values."""
    return {
        "price_dash": int(table["median_thousands"].eq("-").sum()),
        "count_dash": int(table["count_raw"].eq("-").sum()),
        "count_s": int(table["count_raw"].eq("s").sum()),
    }

all_sheet = suppression_counts(raw_sales)
sydney_all_types = sydney_block.loc[sydney_block["ring"].isin(rings) & sydney_block["lga"].ne("Total")]
sydney_lga_markers = suppression_counts(sydney_all_types)
sydney_total_markers = suppression_counts(sydney_all_types.loc[sydney_all_types["source_dwelling"].eq("Total")])
retained_markers = suppression_counts(selected.loc[selected["geo_level"].eq("lga")])
aggregate_markers = suppression_counts(selected.loc[selected["geo_level"].ne("lga")])
export_markers = {
    "price_dash": int(sales["price_status"].eq("suppressed").sum()),
    "count_dash": int(sales["count_status"].eq("suppressed_10_or_fewer").sum()),
    "count_s": int(sales["count_status"].eq("suppressed_30_or_fewer").sum()),
}
assert all_sheet == {"price_dash": 8016, "count_dash": 8016, "count_s": 12992}
assert sydney_lga_markers == {"price_dash": 15, "count_dash": 15, "count_s": 133}
assert sydney_total_markers == {"price_dash": 0, "count_dash": 0, "count_s": 3}
assert aggregate_markers == {"price_dash": 0, "count_dash": 0, "count_s": 0}
assert retained_markers == export_markers == {"price_dash": 15, "count_dash": 15, "count_s": 130}
display(pd.DataFrame.from_dict({
    "T01 whole LGA sheet": all_sheet,
    "T01 Sydney LGA rows": sydney_lga_markers,
    "Removed Sydney LGA Total rows": sydney_total_markers,
    "Retained ring and Sydney aggregate rows": aggregate_markers,
    "Clean export": export_markers,
}, orient="index"))

,price_dash,count_dash,count_s
T01 whole LGA sheet,8016,8016,12992
T01 Sydney LGA rows,15,15,133
Removed Sydney LGA Total rows,0,0,3
Retained ring and Sydney aggregate rows,0,0,0
Clean export,15,15,130


### Check large adjacent-quarter moves

Look for absolute first-published median moves above 25% when the current LGA-quarter has more than 30 published sales. Also count how many have more than 30 recorded sales in the prior quarter. These examples are for source review. They leave the published values in the export.

In [10]:
lga_price_pairs = price_pairs.loc[price_pairs["geo_level"].eq("lga")].copy()
lga_price_pairs["move_pct"] = (
    lga_price_pairs["median_price"] / lga_price_pairs["previous_median_price"] - 1
) * 100
lga_price_pairs["abs_move_pct"] = lga_price_pairs["move_pct"].abs()
large_moves = lga_price_pairs.loc[
    lga_price_pairs["abs_move_pct"].gt(25) & lga_price_pairs["sales_count"].gt(30)
].copy()
both_counts = large_moves.loc[large_moves["previous_sales_count"].gt(30)]
print("Moves above 25% with more than 30 sales")
display(pd.DataFrame({
    "current_quarter": large_moves.groupby("dwelling_type").size(),
    "both_quarters": both_counts.groupby("dwelling_type").size(),
}).reindex(dwellings["dwelling_type"], fill_value=0).fillna(0).astype(int))
print("Five largest moves with more than 30 recorded sales in both quarters")
display(both_counts.nlargest(5, "abs_move_pct")[[
    "quarter_date", "geo_name", "dwelling_type", "previous_median_price",
    "median_price", "previous_sales_count", "sales_count", "move_pct",
]].round({"move_pct": 2}))

Moves above 25% with more than 30 sales


,current_quarter,both_quarters
dwelling_type,,
house,8,7
unit,5,4


Five largest moves with more than 30 recorded sales in both quarters


,quarter_date,geo_name,dwelling_type,previous_median_price,median_price,previous_sales_count,sales_count,move_pct
1053,2022-04-01,Willoughby,unit,1100000,1620000,128,205,47.27
1588,2024-01-01,Woollahra,house,6750000,4100000,105,95,-39.26
1512,2023-10-01,Woollahra,house,4900000,6750000,108,105,37.76
1176,2022-10-01,Lane Cove,house,4575000,2875000,78,57,-37.16
1029,2022-04-01,Mosman,unit,1200000,1600000,80,95,33.33


## 8. Export the two CSVs

Write the validated lookup and sales table in a deterministic order and keep missing cells empty. Export dates as ISO dates and keep the 2025 codes as five characters. The revision estimates and anomaly checks stay in the notebook.

In [11]:
sales_columns = [
    "quarter_date", "geo_level", "geo_name", "region", "dwelling_type",
    "median_price", "sales_count", "price_status", "count_status", "low_sales_flag",
    "source_file", "source_row", *change_columns,
]
export_paths = {
    repo_root / "data/reference/lga_lookup.csv": lga_lookup,
    repo_root / "data/interim/sales_long.csv": sales[sales_columns],
}
for path, table in export_paths.items():
    path.parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(path, index=False, encoding="utf-8", date_format="%Y-%m-%d", lineterminator="\n")
    print(f"Wrote {path.relative_to(repo_root)} with {len(table):,} rows.")

Wrote data/reference/lga_lookup.csv with 34 rows.
Wrote data/interim/sales_long.csv with 2,204 rows.


### Output columns

`lga_lookup.csv`

- `lga` — canonical DCJ local government area name.
- `region` — its DCJ Inner, Middle or Outer Ring.
- `lga_code_2025` — five-character ABS LGA 2025 identifier.
- `abs_name_2025` — ABS label for that code.

`sales_long.csv`

- `quarter_date` — first day of the reference quarter in YYYY-MM-DD format.
- `geo_level` — `lga`, `ring` or `greater_sydney`.
- `geo_name` — canonical LGA, ring or Greater Sydney name.
- `region` — DCJ ring, blank for Greater Sydney.
- `dwelling_type` — `house` for non-strata or `unit` for strata.
- `median_price` — first-published median in Australian dollars, blank when unavailable.
- `sales_count` — published sales count, blank when unavailable.
- `price_status` — `published`, `suppressed` or `not_in_source`.
- `count_status` — `published`, `suppressed_10_or_fewer`, `suppressed_30_or_fewer` or `not_in_source`.
- `low_sales_flag` — true if the count is withheld or the row is absent.
- `source_file` — name of the DCJ workbook for the quarter.
- `source_row` — Excel row number in `LGA`, blank when the row is absent.
- `quarterly_change` — published quarterly change from column K as a decimal fraction, blank if unavailable.
- `annual_change` — published annual change from column L as a decimal fraction, blank if unavailable.

## 9. Measure distance to the Sydney CBD

Use the geometric centroid of each ABS LGA 2025 polygon. The SQL finds it in Australian Albers, then measures straight-line distance along the WGS84 ellipsoid to the Sydney General Post Office. Large rural and bushland areas influence the location of this centroid. The distance represents the whole LGA polygon.

Match the results to the lookup by ABS code. Compare all 34 distances to the two-decimal values from rerunning Part B, C5 of `01_explore.ipynb` and retaining its complete Sydney distance table. DuckDB may need network access on the first run to install its spatial extension.

In [12]:
import duckdb

lookup = pd.read_csv(
    repo_root / "data/reference/lga_lookup.csv", dtype={"lga_code_2025": "string"},
)
distance_sql = (repo_root / "sql/distance.sql").read_text(encoding="utf-8")
with duckdb.connect() as spatial:
    spatial.execute("INSTALL spatial")
    spatial.execute("LOAD spatial")
    # Both the GeoJSON and the distance function use longitude first.
    spatial.execute("SET geometry_always_xy = true")
    distances = spatial.execute(distance_sql, {
        "boundaries_path": str(raw_dir / "abs_asgs/LGA_2025_Sydney_region.geojson"),
        "gpo_path": str(raw_dir / "abs_asgs/Sydney_GPO_00763.geojson"),
    }).df()

distances["lga_code_2025"] = distances["lga_code_2025"].astype("string")

lga_keys = ["lga", "lga_code_2025"]
distance_check = lookup[lga_keys].merge(distances, on="lga_code_2025", how="left", validate="one_to_one")
t02_distance_km = {
    "10500": 9.81,
    "10750": 34.79,
    "10900": 76.16,
    "11300": 9.83,
    "11450": 47.36,
    "11500": 41.11,
    "11520": 8.74,
    "11570": 16.93,
    "11650": 62.52,
    "12380": 20.67,
    "12850": 28.90,
    "12930": 16.03,
    "13800": 71.57,
    "14000": 33.35,
    "14100": 6.94,
    "14170": 5.54,
    "14500": 15.68,
    "14700": 6.51,
    "14900": 36.18,
    "15350": 5.49,
    "15950": 3.88,
    "15990": 20.08,
    "16260": 18.17,
    "16350": 46.73,
    "16550": 8.89,
    "16700": 11.78,
    "17100": 12.23,
    "17150": 27.18,
    "17200": 2.45,
    "17420": 38.48,
    "18050": 6.00,
    "18250": 7.47,
    "18400": 75.22,
    "18500": 4.62,
}
assert set(t02_distance_km) == set(lookup["lga_code_2025"])
distance_check["t02_distance_km"] = distance_check["lga_code_2025"].map(t02_distance_km)
distance_check["abs_difference_km"] = (
    distance_check["distance_km"] - distance_check["t02_distance_km"]
).abs()
assert distance_check[["centroid_lat", "centroid_lon", "abs_difference_km"]].notna().all().all()
assert distance_check["abs_difference_km"].le(0.01 + 1e-9).all()
print(f"Compared all {len(distance_check)} LGAs with T02.")
print(f"Maximum absolute distance difference: {distance_check['abs_difference_km'].max():.2f} km.")
with pd.option_context("display.float_format", "{:.2f}".format):
    display(distance_check[lga_keys + ["distance_km", "t02_distance_km", "abs_difference_km"]])

Compared all 34 LGAs with T02.
Maximum absolute distance difference: 0.00 km.


,lga,lga_code_2025,distance_km,t02_distance_km,abs_difference_km
0,Bayside,10500,9.81,9.81,0.00
1,Blacktown,10750,34.79,34.79,0.00
2,Blue Mountains,10900,76.16,76.16,0.00
3,Burwood,11300,9.83,9.83,0.00
4,Camden,11450,47.36,47.36,0.00
5,Campbelltown,11500,41.11,41.11,0.00
6,Canada Bay,11520,8.74,8.74,0.00
7,Canterbury-Bankstown,11570,16.93,16.93,0.00
8,Central Coast,11650,62.52,62.52,0.00
9,Cumberland,12380,20.67,20.67,0.00


## 10. Calculate population growth from June 2016 to June 2019

Use the ABS estimated resident population at 30 June 2016 and 30 June 2019. Both years are on LGA 2025 boundaries, so the counts are comparable by code. Divide the 2019 count by the 2016 count and subtract one.

In [13]:
population = pd.read_excel(
    raw_dir / "abs_erp/32180DS0004_2001-25.xlsx", sheet_name="Table 1",
    skiprows=4, usecols="A,R,U", dtype="string",
)
assert list(population.columns[1:]) == [2016, 2019]
population.columns = ["lga_code_2025", "pop_2016", "pop_2019"]
population = population.loc[population["lga_code_2025"].isin(lookup["lga_code_2025"])].copy()
for column in ["pop_2016", "pop_2019"]:
    population[column] = parse_number(population[column]).astype("Int64")
    assert population[column].gt(0).all()
population["pop_growth_2016_19"] = population["pop_2019"] / population["pop_2016"] - 1
print(f"Read two June population estimates for each of {len(population)} LGAs.")

Read two June population estimates for each of 34 LGAs.


## 11. Attach 2016 household income where the code matches

Join the 2016 NSW Census G02 medians to the 2016 Census labels using their published codes. Compare those codes with the LGA 2025 lookup. A code match identifies the row to use. It allows for changes in spelling and can still include small boundary changes.

The 2016 Census preceded the creation of Bayside. Show the published Botany Bay and Rockdale medians alongside their original codes for reference. The merged area's median remains blank because separate medians cannot be combined into a new median.

In [14]:
census_dir = raw_dir / "abs_census"
income = pd.read_csv(
    census_dir / "2016Census_G02_NSW_LGA.csv",
    usecols=["LGA_CODE_2016", "Median_tot_hhd_inc_weekly"],
    dtype={"LGA_CODE_2016": "string"},
).rename(columns={
    "LGA_CODE_2016": "census_code_2016",
    "Median_tot_hhd_inc_weekly": "income_2016_weekly",
})
income["income_2016_weekly"] = parse_number(income["income_2016_weekly"]).astype("Int64")
descriptor = pd.read_excel(
    census_dir / "2016Census_geog_desc_1st_and_2nd_release.xlsx",
    sheet_name="2016_ASGS_Non-ABS_Structures",
    usecols=["ASGS_Structure", "Census_Code_2016", "ASGS_Code_2016", "Census_Name_2016"],
    dtype="string",
)
labels_2016 = descriptor.loc[descriptor["ASGS_Structure"].eq("LGA")].drop(columns="ASGS_Structure")
labels_2016 = labels_2016.rename(columns={
    "Census_Code_2016": "census_code_2016",
    "ASGS_Code_2016": "lga_code_2016",
    "Census_Name_2016": "census_name_2016",
})
income = income.merge(labels_2016, on="census_code_2016", how="left", validate="one_to_one")
predecessor_codes = ["11100", "16650"]
relevant_codes = set(lookup["lga_code_2025"]) | set(predecessor_codes)
income = income.loc[income["census_code_2016"].str.removeprefix("LGA").isin(relevant_codes)]
assert income["census_code_2016"].eq("LGA" + income["lga_code_2016"]).all()

income_by_lga = lookup[lga_keys].merge(
    income, left_on="lga_code_2025", right_on="lga_code_2016", how="left", validate="one_to_one",
)
income_matches = income_by_lga.dropna(subset=["lga_code_2016"])
assert len(income_matches) == 33
assert income_matches["lga_code_2016"].eq(income_matches["lga_code_2025"]).all()
assert income_by_lga.loc[income_by_lga["lga_code_2016"].isna(), "lga"].tolist() == ["Bayside"]
print("33 matching codes with the canonical DCJ label and the published 2016 Census label")
display(income_matches[lga_keys + ["lga_code_2016", "census_name_2016", "income_2016_weekly"]])

predecessors = income.loc[income["lga_code_2016"].isin(predecessor_codes)].sort_values("lga_code_2016")
assert predecessors["lga_code_2016"].tolist() == predecessor_codes
print("Bayside predecessor medians in dollars per week")
display(predecessors[["lga_code_2016", "census_name_2016", "income_2016_weekly"]])

33 matching codes with the canonical DCJ label and the published 2016 Census label


,lga,lga_code_2025,lga_code_2016,census_name_2016,income_2016_weekly
1,Blacktown,10750,10750,Blacktown (C),1711
2,Blue Mountains,10900,10900,Blue Mountains (C),1468
3,Burwood,11300,11300,Burwood (A),1569
4,Camden,11450,11450,Camden (A),2047
5,Campbelltown,11500,11500,Campbelltown (C) (NSW),1459
6,Canada Bay,11520,11520,Canada Bay (A),2061
7,Canterbury-Bankstown,11570,11570,Canterbury-Bankstown (A),1298
8,Central Coast,11650,11650,Central Coast (C) (NSW),1258
9,Cumberland,12380,12380,Cumberland (A),1379
10,Fairfield,12850,12850,Fairfield (C),1222


Bayside predecessor medians in dollars per week


,lga_code_2016,census_name_2016,income_2016_weekly
13,11100,Botany Bay (C),1626
96,16650,Rockdale (C),1575


## 12. Average 2019 median weekly rents by LGA and dwelling type

Use column H of Table 1 for `House` and `Flat/Unit`, with `Total` bedrooms, within DCJ's three Sydney rings. For each LGA and dwelling type, take the simple mean of available quarterly medians and count how many went into it. A suppressed bond count in column J can accompany a published median in H, which still goes into the mean.

Issue 129 is the September quarter. Its September filename, cover and `LGA!F1` identify the correct period. `LGA!A1` repeats June.

In [15]:
rent_issues = {
    "2019-01-01": "issue-127-rent-tables-march-2019.xlsx",
    "2019-04-01": "issue-128-rent-tables-june-2019.xlsx",
    "2019-07-01": "issue-129-rent-tables-september-2019.xlsx",
    "2019-10-01": "issue-130-rent-tables-december-2019.xlsx",
}
rent_types = {"House": "house", "Flat/Unit": "unit"}

def read_rent_workbook(path, quarter_date):
    """Keep the Sydney LGA median rent and the source row for each dwelling type."""
    workbook = load_workbook(path, read_only=True, data_only=True)
    try:
        rows = list(workbook["LGA"].iter_rows(max_col=10, values_only=True))
    finally:
        workbook.close()
    heading = next(index for index, row in enumerate(rows) if row[4] == "Dwelling Types")
    data = []
    for row_number, row in enumerate(rows[heading + 1:], start=heading + 2):
        gmr, area, ring, lga, dwelling, bedrooms = row[:6]
        if (
            gmr == "GMR (Greater Metropolitan Region)" and area == "Greater Sydney"
            and ring in rings and lga != "Total" and dwelling in rent_types and bedrooms == "Total"
        ):
            data.append((lga, ring, rent_types[dwelling], row[7], row[9], row_number))
    table = pd.DataFrame(
        data, columns=["lga", "region", "dwelling_type", "rent_raw", "new_bonds", "source_row"],
    )
    table["quarter_date"] = quarter_date
    return table

rent_tables = [
    read_rent_workbook(raw_dir / "dcj_rent" / filename, date) for date, filename in rent_issues.items()
]
rent_source = pd.concat(rent_tables, ignore_index=True)
assert rent_source["region"].eq(rent_source["lga"].map(lookup.set_index("lga")["region"])).all()
rent_keys = ["lga", "dwelling_type", "quarter_date"]
rent_quarters = pd.DataFrame({"quarter_date": list(rent_issues)})
rent_grid = lookup[["lga"]].merge(dwellings, how="cross").merge(rent_quarters, how="cross")
rent_rows = rent_grid.merge(rent_source, on=rent_keys, how="left", validate="one_to_one")
rent_rows["median_weekly_rent"] = parse_number(rent_rows["rent_raw"])
rent_rows["source_file"] = rent_rows["quarter_date"].map(rent_issues)

rent_by_lga = lookup[["lga"]].copy()
for dwelling_type in rent_types.values():
    dwelling_rows = rent_rows.loc[rent_rows["dwelling_type"].eq(dwelling_type)]
    dwelling_summary = dwelling_rows.groupby("lga")["median_weekly_rent"].agg(["mean", "count"])
    dwelling_summary.columns = [
        f"rent_2019_{dwelling_type}_weekly", f"rent_2019_quarters_used_{dwelling_type}",
    ]
    rent_by_lga = rent_by_lga.merge(dwelling_summary, on="lga", validate="one_to_one")

missing_rents = rent_rows.loc[rent_rows["median_weekly_rent"].isna()]
print(f"Quarterly rent medians available: {len(rent_rows) - len(missing_rents)} of {len(rent_rows)}.")
if not missing_rents.empty:
    display(missing_rents[rent_keys + ["source_file", "source_row", "rent_raw"]])
suppressed_bond_count = rent_rows["new_bonds"].isin(["-", "s"])
print("Published rent medians with a suppressed bond count:",
      (suppressed_bond_count & rent_rows["median_weekly_rent"].notna()).sum())
rent_checks = rent_rows.loc[
    (rent_rows["lga"].eq("Blacktown") & rent_rows["dwelling_type"].eq("house"))
    | (rent_rows["lga"].eq("Sydney") & rent_rows["dwelling_type"].eq("unit"))
]
print("Quarterly inputs for Blacktown houses and Sydney units")
display(rent_checks[rent_keys + ["median_weekly_rent", "source_file", "source_row"]])

Quarterly rent medians available: 272 of 272.
Published rent medians with a suppressed bond count: 6
Quarterly inputs for Blacktown houses and Sydney units


,lga,dwelling_type,quarter_date,median_weekly_rent,source_file,source_row
8,Blacktown,house,2019-01-01,470,issue-127-rent-tables-march-2019.xlsx,21629
9,Blacktown,house,2019-04-01,450,issue-128-rent-tables-june-2019.xlsx,21646
10,Blacktown,house,2019-07-01,460,issue-129-rent-tables-september-2019.xlsx,21629
11,Blacktown,house,2019-10-01,460,issue-130-rent-tables-december-2019.xlsx,21633
228,Sydney,unit,2019-01-01,650,issue-127-rent-tables-march-2019.xlsx,21062
229,Sydney,unit,2019-04-01,650,issue-128-rent-tables-june-2019.xlsx,21079
230,Sydney,unit,2019-07-01,650,issue-129-rent-tables-september-2019.xlsx,21062
231,Sydney,unit,2019-10-01,640,issue-130-rent-tables-december-2019.xlsx,21065


## 13. Validate and export the LGA factors

Keep the lookup order and check that each LGA and code appears once. Verify the permitted gaps, the ranges and the source spot checks. The Camden population growth is a confirmed exception to the 30% sanity limit. Keep the published counts and flag any out-of-range result for review.

Write the distance to two decimal places and growth as a decimal fraction to four places. Leave any unavailable values empty.

In [16]:
factors = lookup[lga_keys].copy()
distance_values = distance_check[["lga_code_2025", "distance_km"]]
income_values = income_by_lga[["lga_code_2025", "income_2016_weekly"]]
for table in [distance_values, population, income_values]:
    factors = factors.merge(table, on="lga_code_2025", how="left", validate="one_to_one")
factors = factors.merge(rent_by_lga, on="lga", how="left", validate="one_to_one")

factor_columns = lga_keys + [
    "distance_km", "pop_2016", "pop_2019", "pop_growth_2016_19", "income_2016_weekly",
    "rent_2019_house_weekly", "rent_2019_unit_weekly",
    "rent_2019_quarters_used_house", "rent_2019_quarters_used_unit",
]
factors = factors[factor_columns]
assert len(factors) == 34
pd.testing.assert_frame_equal(factors[lga_keys], lookup[lga_keys], check_dtype=False)
for key in lga_keys:
    assert factors[key].is_unique and factors[key].notna().all()
required = lga_keys + ["distance_km", "pop_2016", "pop_2019", "pop_growth_2016_19"]
assert factors[required].notna().all().all()
assert factors["income_2016_weekly"].isna().eq(factors["lga"].eq("Bayside")).all()
assert factors["distance_km"].between(0, 100).all()
assert factors["income_2016_weekly"].dropna().between(800, 5_000).all()

for dwelling_type in rent_types.values():
    rent = factors[f"rent_2019_{dwelling_type}_weekly"]
    quarters_used = factors[f"rent_2019_quarters_used_{dwelling_type}"]
    assert quarters_used.notna().all() and quarters_used.between(0, 4).all()
    assert rent.isna().eq(quarters_used.eq(0)).all()
    assert rent.dropna().between(250, 2_000).all()
    zero_quarter_lgas = factors.loc[quarters_used.eq(0), "lga"].tolist()
    print(f"{dwelling_type.capitalize()} rent with no usable quarters: {zero_quarter_lgas or 'none'}.")

by_lga = factors.set_index("lga")
assert by_lga.loc["Blacktown", ["pop_2016", "pop_2019"]].tolist() == [348_030, 380_186]
assert round(by_lga.loc["Blacktown", "pop_growth_2016_19"], 4) == 0.0924
assert by_lga.loc["Sydney", ["pop_2016", "pop_2019"]].tolist() == [222_717, 229_596]
assert by_lga.loc[["Blacktown", "Blue Mountains", "Sydney"], "distance_km"].tolist() == [34.79, 76.16, 2.45]
march_rents = rent_rows.loc[rent_rows["quarter_date"].eq("2019-01-01")].set_index(["lga", "dwelling_type"])
assert march_rents.loc[("Blacktown", "house"), "median_weekly_rent"] == 470
assert march_rents.loc[("Sydney", "unit"), "median_weekly_rent"] == 650

growth_in_range = factors["pop_growth_2016_19"].between(-0.10, 0.30)
growth_exceptions = factors.loc[~growth_in_range, ["lga", "pop_2016", "pop_2019", "pop_growth_2016_19"]]
assert growth_exceptions["lga"].tolist() == ["Camden"]
assert by_lga.loc["Camden", ["pop_2016", "pop_2019"]].tolist() == [80_268, 104_706]
print(f"Population growth in the −10% to +30% range: {growth_in_range.sum()} of {len(factors)}.")
print("Source-confirmed exception from ERP Table 1, cells R27 and U27")
display(growth_exceptions)

export_factors = factors.copy()
export_factors["distance_km"] = factors["distance_km"].map("{:.2f}".format)
export_factors["pop_growth_2016_19"] = factors["pop_growth_2016_19"].map("{:.4f}".format)
factor_path = repo_root / "data/interim/lga_factors.csv"
export_factors.to_csv(factor_path, index=False, encoding="utf-8", lineterminator="\n")
print(f"Wrote {factor_path.relative_to(repo_root)} with {len(factors)} rows and {len(factor_columns)} columns.")
print("Keys, missing-value rules, spot checks and the documented range exception verified.")
with pd.option_context("display.max_columns", None):
    display(export_factors)

House rent with no usable quarters: none.
Unit rent with no usable quarters: none.
Population growth in the −10% to +30% range: 33 of 34.
Source-confirmed exception from ERP Table 1, cells R27 and U27


,lga,pop_2016,pop_2019,pop_growth_2016_19
4,Camden,80268,104706,0.304455


Wrote data/interim/lga_factors.csv with 34 rows and 11 columns.
Keys, missing-value rules, spot checks and the documented range exception verified.


,lga,lga_code_2025,distance_km,pop_2016,pop_2019,pop_growth_2016_19,income_2016_weekly,rent_2019_house_weekly,rent_2019_unit_weekly,rent_2019_quarters_used_house,rent_2019_quarters_used_unit
0,Bayside,10500,9.81,164534,173913,0.0570,<NA>,647.0,570.0,4,4
1,Blacktown,10750,34.79,348030,380186,0.0924,1711,460.0,370.0,4,4
2,Blue Mountains,10900,76.16,78835,79077,0.0031,1468,442.5,305.0,4,4
3,Burwood,11300,9.83,38536,40333,0.0466,1569,667.5,548.75,4,4
4,Camden,11450,47.36,80268,104706,0.3045,2047,500.75,344.5,4,4
5,Campbelltown,11500,41.11,161562,172286,0.0664,1459,422.5,353.25,4,4
6,Canada Bay,11520,8.74,92534,92609,0.0008,2061,745.0,585.0,4,4
7,Canterbury-Bankstown,11570,16.93,361862,375333,0.0372,1298,550.0,411.25,4,4
8,Central Coast,11650,62.52,336611,345695,0.0270,1258,442.5,351.25,4,4
9,Cumberland,12380,20.67,225691,240090,0.0638,1379,497.5,427.5,4,4


## 14. SQL transform and export

Run the transformation in a fresh in-memory DuckDB connection. The SQL checks that every series has 29 consecutive quarter starts before it uses row offsets. It then builds the four tables, keeping the published prices as the main series and the revised estimates for the D08 sensitivity check.

In [17]:
from contextlib import chdir

transform_sql = (repo_root / "sql/transform.sql").read_text(encoding="utf-8")
warehouse = duckdb.connect(":memory:")
with chdir(repo_root):
    warehouse.execute(transform_sql)
print("Built all four SQL tables after validating the quarterly sales grid.")

Built all four SQL tables after validating the quarterly sales grid.


### Preview the four tables

Show the row count and the first five rows of each table. Read the rows in the same order that will be used for export.

In [18]:
output_keys = {
    "fact_sales": ["lga", "quarter_date", "dwelling_type"],
    "dim_lga": ["lga"],
    "dim_date": ["quarter_date"],
    "lga_growth": ["lga", "dwelling_type"],
}
export_order = {name: ", ".join(columns) for name, columns in output_keys.items()}
export_order["dim_lga"] = "sort_order"
sql_tables = {}
for name, order in export_order.items():
    table = warehouse.sql(f"SELECT * FROM {name} ORDER BY {order}").df()
    sql_tables[name] = table
    print(f"{name}: {len(table):,} rows")
    with pd.option_context("display.max_columns", None):
        display(table.head())

fact_sales: 2,204 rows


,lga,quarter_date,dwelling_type,median_price,sales_count,price_status,low_sales_flag,yoy_change,median_price_revised_est,revised_source
0,Bayside,2019-01-01,house,1138000,98,published,False,NaN,1150000,annual
1,Bayside,2019-01-01,unit,675000,265,published,False,NaN,685000,annual
2,Bayside,2019-04-01,house,1127000,174,published,False,NaN,1160000,annual
3,Bayside,2019-04-01,unit,690000,442,published,False,NaN,700000,annual
4,Bayside,2019-07-01,house,1250000,162,published,False,NaN,1251000,annual


dim_lga: 38 rows


,lga,geo_level,region,lga_code_2025,distance_km,pop_growth_2016_19,income_2016_weekly,rent_2019_house_weekly,rent_2019_unit_weekly,sort_order
0,Greater Sydney,greater_sydney,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,1
1,Inner Ring,ring,Inner Ring,NaN,NaN,NaN,<NA>,NaN,NaN,2
2,Middle Ring,ring,Middle Ring,NaN,NaN,NaN,<NA>,NaN,NaN,3
3,Outer Ring,ring,Outer Ring,NaN,NaN,NaN,<NA>,NaN,NaN,4
4,Bayside,lga,Middle Ring,10500,9.81,0.057,<NA>,647.0,570.0,5


dim_date: 29 rows


,quarter_date,year,quarter_label,q_index,is_baseline_2019,is_latest_4q
0,2019-01-01,2019,2019 Q1,1,True,False
1,2019-04-01,2019,2019 Q2,2,True,False
2,2019-07-01,2019,2019 Q3,3,True,False
3,2019-10-01,2019,2019 Q4,4,True,False
4,2020-01-01,2020,2020 Q1,5,False,False


lga_growth: 76 rows


,lga,geo_level,region,dwelling_type,baseline_quarters,latest_quarters,baseline_low_sales_quarters,latest_low_sales_quarters,price_2019,price_latest,growth_pct,growth_annualised,price_2019_rev,price_latest_rev,growth_pct_rev,growth_note
0,Bayside,lga,Middle Ring,house,4,4,0,0,1220250.0,1986000.0,0.627535,0.081048,1232250.0,1997250.0,0.620816,NaN
1,Bayside,lga,Middle Ring,unit,4,4,0,0,702500.0,818750.0,0.165480,0.024804,707500.0,819250.0,0.157951,NaN
2,Blacktown,lga,Outer Ring,house,4,4,0,0,716000.0,1236250.0,0.726606,0.091317,715000.0,1235000.0,0.727273,NaN
3,Blacktown,lga,Outer Ring,unit,4,4,0,0,533250.0,673500.0,0.263010,0.038066,540000.0,680000.0,0.259259,NaN
4,Blue Mountains,lga,Outer Ring,house,4,4,0,0,652500.0,1038500.0,0.591571,0.077190,652500.0,1035500.0,0.586973,NaN


### Validate keys, windows and price calculations

Check table sizes, unique keys, fact-to-dimension matches and the two four-quarter windows. Recompute three YoY changes and three window-to-window growth rates in pandas using the input CSV. List all missing growth values and count how each revised estimate was obtained.

In [19]:
expected_rows = {"fact_sales": 2204, "dim_lga": 38, "dim_date": 29, "lga_growth": 76}
for name, table in sql_tables.items():
    assert len(table) == expected_rows[name], (name, len(table))
    table_keys = output_keys[name]
    assert table[table_keys].notna().all().all(), name
    assert not table.duplicated(table_keys).any(), name

fact = sql_tables["fact_sales"]
geo_dim = sql_tables["dim_lga"]
date_dim = sql_tables["dim_date"]
growth = sql_tables["lga_growth"]
assert set(fact["lga"]) <= set(geo_dim["lga"])
assert set(fact["quarter_date"]) <= set(date_dim["quarter_date"])
assert date_dim["is_baseline_2019"].sum() == 4
assert date_dim["is_latest_4q"].sum() == 4
print("Row counts, unique keys, foreign keys and both four-quarter windows passed.")

sales_csv = pd.read_csv(repo_root / "data/interim/sales_long.csv", parse_dates=["quarter_date"])
latest_dates = sales_csv["quarter_date"].drop_duplicates().sort_values().tail(4)
last_date = latest_dates.iloc[-1]
prior_year_date = last_date - pd.DateOffset(years=1)
assert date_dim.loc[date_dim["is_latest_4q"], "quarter_date"].tolist() == latest_dates.tolist()
assert date_dim["is_baseline_2019"].eq(date_dim["quarter_date"].dt.year.eq(2019)).all()

spot_series = [("Blacktown", "house"), ("Sydney", "unit"), ("Greater Sydney", "house")]
fact_lookup = fact.set_index(["lga", "dwelling_type", "quarter_date"])
growth_lookup = growth.set_index(["lga", "dwelling_type"])
yoy_checks = []
growth_checks = []
for name, dwelling in spot_series:
    series = sales_csv.loc[
        sales_csv["geo_name"].eq(name) & sales_csv["dwelling_type"].eq(dwelling)
    ].set_index("quarter_date")["median_price"]

    yoy_pandas = series.loc[last_date] / series.loc[prior_year_date] - 1
    yoy_sql = fact_lookup.loc[(name, dwelling, last_date), "yoy_change"]
    assert abs(yoy_sql - yoy_pandas) < 1e-9, (name, dwelling, yoy_sql, yoy_pandas)
    yoy_checks.append((name, dwelling, last_date.date(), yoy_sql, yoy_pandas))

    baseline = series.loc[series.index.year == 2019]
    latest = series.loc[latest_dates]
    assert baseline.count() == latest.count() == 4
    growth_pandas = latest.mean() / baseline.mean() - 1
    growth_sql = growth_lookup.loc[(name, dwelling), "growth_pct"]
    assert abs(growth_sql - growth_pandas) < 1e-9, (name, dwelling, growth_sql, growth_pandas)
    growth_checks.append((name, dwelling, growth_sql, growth_pandas, abs(growth_sql - growth_pandas)))

with pd.option_context("display.float_format", "{:.12f}".format):
    print("YoY checks against one year earlier")
    display(pd.DataFrame(yoy_checks, columns=["lga", "dwelling_type", "quarter_date", "sql", "pandas"]))
    print("Growth checks using the two four-quarter windows")
    display(pd.DataFrame(growth_checks, columns=["lga", "dwelling_type", "sql", "pandas", "abs_diff"]))

missing_growth = growth.loc[growth["growth_pct"].isna(), ["lga", "dwelling_type", "growth_note"]]
assert growth["growth_note"].notna().eq(growth["growth_pct"].isna()).all()
assert pd.isna(growth_lookup.loc[("Wollondilly", "unit"), "growth_pct"])
wollondilly_note = growth_lookup.loc[("Wollondilly", "unit"), "growth_note"]
assert wollondilly_note.startswith("baseline: 0 of 4 quarters published")
print("All rows with missing first-published growth")
display(missing_growth)

revision_counts = fact["revised_source"].fillna("null").value_counts()
assert set(revision_counts.index) <= {"annual", "quarterly", "first_published", "null"}
assert fact["revised_source"].isna().eq(fact["median_price_revised_est"].isna()).all()
latest_sources = fact.loc[fact["quarter_date"].eq(last_date), "revised_source"]
assert len(latest_sources) == 76 and latest_sources.eq("first_published").all()
print("Revised estimate source for all sales rows")
display(revision_counts.rename("rows"))
print(f"All {len(latest_sources)} rows in the latest quarter use first-published medians.")

for column in ["median_price", "median_price_revised_est"]:
    assert fact[column].dropna().gt(0).all(), column
for column in ["price_2019", "price_latest", "price_2019_rev", "price_latest_rev"]:
    assert growth[column].dropna().gt(0).all(), column
for column in ["growth_pct", "growth_annualised", "growth_pct_rev"]:
    assert growth[column].dropna().between(-0.50, 2.00).all(), column

baseline_indexes = date_dim.loc[date_dim["is_baseline_2019"], "q_index"]
latest_indexes = date_dim.loc[date_dim["is_latest_4q"], "q_index"]
window_years = (latest_indexes.mean() - baseline_indexes.mean()) / 4
expected_annualised = (1 + growth["growth_pct"]) ** (1 / window_years) - 1
assert (growth["growth_annualised"] - expected_annualised).dropna().abs().lt(1e-9).all()
print(f"Window midpoint gap: {window_years:g} years.")
print("All price and growth range checks passed.")

Row counts, unique keys, foreign keys and both four-quarter windows passed.


YoY checks against one year earlier


,lga,dwelling_type,quarter_date,sql,pandas
0,Blacktown,house,2026-01-01,0.069289991446,0.069289991446
1,Sydney,unit,2026-01-01,-0.013265306122,-0.013265306122
2,Greater Sydney,house,2026-01-01,0.083916083916,0.083916083916


Growth checks using the two four-quarter windows


,lga,dwelling_type,sql,pandas,abs_diff
0,Blacktown,house,0.726606145251,0.726606145251,0.000000000000
1,Sydney,unit,0.068946497518,0.068946497518,0.000000000000
2,Greater Sydney,house,0.674695534506,0.674695534506,0.000000000000


All rows with missing first-published growth


,lga,dwelling_type,growth_note
73,Wollondilly,unit,baseline: 0 of 4 quarters published


Revised estimate source for all sales rows


revised_source
annual             1888
quarterly           232
first_published      78
null                  6
Name: rows, dtype: int64

All 76 rows in the latest quarter use first-published medians.
Window midpoint gap: 6.25 years.
All price and growth range checks passed.


### Export the four Power BI CSVs

Write the tables in a fixed order. DuckDB writes the headers, ISO dates, lowercase booleans and full-precision numbers. Missing values are empty fields.

In [20]:
processed_dir = repo_root / "data/processed"
processed_dir.mkdir(parents=True, exist_ok=True)
for name, order in export_order.items():
    output_path = processed_dir / f"{name}.csv"
    warehouse.execute(
        f"COPY (SELECT * FROM {name} ORDER BY {order}) TO ? (FORMAT CSV, HEADER, DELIMITER ',')",
        [str(output_path)],
    )
    print(f"Wrote {output_path.relative_to(repo_root)} with {len(sql_tables[name]):,} rows.")
warehouse.close()

Wrote data/processed/fact_sales.csv with 2,204 rows.
Wrote data/processed/dim_lga.csv with 38 rows.
Wrote data/processed/dim_date.csv with 29 rows.
Wrote data/processed/lga_growth.csv with 76 rows.
